# BirdCLEF+ 2026
###### Authors: E. Macedo, G. Cruz, L. Medina  

## Setup
---
### Summary

You must first download your api keys from Kaggle. In the website, go to `Settings` -> `API Tokens` and click on `Create Legacy API Key`. This will generate a `kaggle.json` file needed to validate your Kaggle access.

#### Why not use the `API Tokens`?

By using the `kaggle.json` file, we avoid having to store credentials in the actual repository. This is a bit of a more cumbersome approach, but for the time being it is a generalizable approach.


----
### Installing and Authenticating Kaggle

In [ ]:
!pip install -q kaggle
from google.colab import files
_ = files.upload() # Mute output

In the prompt, upload your Kaggle auth (kaggle.json)

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)

!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

### Downloading BirdCLEF data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('birdclef-2026')

100%|██████████| 15.0G/15.0G [02:51<00:00, 93.7MB/s]

Extracting files...


In [ ]:
!ls /root/.cache/kagglehub/competitions/birdclef-2026

recording_location.txt	test_soundscapes  train_soundscapes
sample_submission.csv	train_audio	  train_soundscapes_labels.csv
taxonomy.csv		train.csv


### Evaluation metric

As defined by the organisation [here](https://www.kaggle.com/code/metric/birdclef-roc-auc).

The evaluation metric for this contest is a version of macro-averaged ROC-AUC that skips classes that have no true positive labels.

Removed call to kaggle_metric_utilities to reduce dependencies.

In [1]:
import pandas as pd
import pandas.api.types

import sklearn.metrics


class ParticipantVisibleError(Exception):
    pass


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    '''
    Version of macro-averaged ROC-AUC score that ignores all classes that have no true positive labels.
    '''
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    if not pandas.api.types.is_numeric_dtype(submission.values):
        bad_dtypes = {x: submission[x].dtype  for x in submission.columns if not pandas.api.types.is_numeric_dtype(submission[x])}
        raise ParticipantVisibleError(f'Invalid submission data types found: {bad_dtypes}')

    solution_sums = solution.sum(axis=0)
    scored_columns = list(solution_sums[solution_sums > 0].index.values)
    assert len(scored_columns) > 0

    return sklearn.metrics.roc_auc_score(solution[scored_columns].values, submission[scored_columns].values, average='macro')